**Recommendation**

In [1]:
import pandas as pd

gid2prot = pd.read_csv('/data/qingyuyang/ESM4SL/data/mapping/gid2protein.csv')

gene_pd = pd.read_csv('/data/qingyuyang/ESM4SL/data/mapping/meta_table_9845.csv')
gid2gn = dict(zip(list(gene_pd['entrez_id']), list(gene_pd['symbol'])))
gn2gid = {value: key for key, value in gid2gn.items()}

In [2]:
import pandas as pd
target_gid = gn2gid['AKT1']
print(target_gid)

a549 = pd.read_csv('/data/qingyuyang/ESM4SL/data/SLKB/cell_line/table/A549.csv')
a549_target = a549[(a549['0'] == target_gid) | (a549['1'] == target_gid)]
a549_others = a549[(a549['0'] != target_gid) & (a549['1'] != target_gid)]
print(a549_target)

207
        0      1  2
68    207  10000  0
69    207   6195  1
70    207   6196  0
71    207   8986  1
72    207   9252  1
...   ...    ... ..
5545  207   3614  0
5546  207   7314  0
5547  207   7316  0
5548  207   5595  0
5549  207  10038  0

[93 rows x 3 columns]


In [4]:
from utils import *

out_path = '/data/qingyuyang/ESM4SL/data/SLKB/case/A549/AKT1'
os.makedirs(out_path, exist_ok=True)
for i in range(5):
    a549_target.to_csv(f'{out_path}/sl_test_{i}.csv', index=False)

def split_case(df: pd.DataFrame, save_path: Path):
    """ Modified from C2 """
    genes = np.array(list(set(df['0']) | set(df['1'])))
    sl_pairs = df.to_numpy()

    index = 0
    # kf = KFold(n_splits=5, shuffle=True, random_state=42)

    # for train_index, test_index in kf.split(genes):
    # train_genes = genes[train_index]
    # test_genes = genes[test_index]

    # split train/valid from train gene
    kf_train_valid = KFold(n_splits=5, shuffle=True, random_state=42)
    for train_index, valid_index in kf_train_valid.split(genes):
        train_genes = genes[train_index]
        valid_genes = genes[valid_index]
        # break

        train_data = get_pairs(sl_pairs, train_genes, test_genes=None, type=1)
        valid_c2_data = get_pairs(sl_pairs, train_genes, test_genes=valid_genes, type=2)
        # test_c2_data = get_pairs(sl_pairs, train_genes, test_genes, type=2)

        train_data = numpy2df(train_data)
        valid_c2_data = numpy2df(valid_c2_data)
        # test_c2_data = numpy2df(test_c2_data)
        assert are_dataframes_subset(pd.concat([train_data, valid_c2_data], ignore_index=True), df)
        # test_c2_data = check_C2_property(test_c2_data, train_data)  #pd.concat([train_data, valid_c2_data], ignore_index=True)
        valid_c2_data = check_C2_property(valid_c2_data, train_data)

        train_data.to_csv(f'{save_path}/sl_train_{index}.csv', index=False)
        valid_c2_data.to_csv(f'{save_path}/sl_val_{index}.csv', index=False)
        # test_c2_data.to_csv(f'{save_path}/sl_test_{index}.csv', index=False)
        index += 1

split_case(a549_others, out_path)

In [1]:
a549 = pd.read_csv('/home/qingyuyang/ESM4SL/data/SLKB/cell_line/table/A549.csv')
a549_is_sl = a549[a549['2'] == 1]
a549_is_sl

,0,1,2
5,25,1445,1
40,8038,101,1
41,8038,8754,1
43,8728,101,1
44,8728,8754,1
...,...,...,...
5109,3098,7086,1
5303,4199,5230,1
5439,5213,5226,1
5452,5214,5226,1


In [14]:
temp_id = gn2gid['RPS6KA1']
print(temp_id)
print(gid2prot[gid2prot['From'] == temp_id]['Protein names'].item())
print(gid2prot[gid2prot['From'] == temp_id]['Gene Names'].item())

6195
Ribosomal protein S6 kinase alpha-1 (S6K-alpha-1) (EC 2.7.11.1) (90 kDa ribosomal protein S6 kinase 1) (p90-RSK 1) (p90RSK1) (p90S6K) (MAP kinase-activated protein kinase 1a) (MAPK-activated protein kinase 1a) (MAPKAP kinase 1a) (MAPKAPK-1a) (Ribosomal S6 kinase 1) (RSK-1)
RPS6KA1 MAPKAPK1A RSK1


In [11]:
temp_id = 207
selected = a549_is_sl[(a549_is_sl['0'] == temp_id) | (a549_is_sl['1'] == temp_id)]
selected['0'] = selected['0'].map(gid2gn)
selected['1'] = selected['1'].map(gid2gn)
selected

/tmp/ipykernel_3790116/705229856.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected['0'] = selected['0'].map(gid2gn)
/tmp/ipykernel_3790116/705229856.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected['1'] = selected['1'].map(gid2gn)


,0,1,2
69,AKT1,RPS6KA1,1
71,AKT1,RPS6KA4,1
72,AKT1,RPS6KA5,1
74,AKT1,RPS6KB2,1
76,AKT1,SGK2,1
2135,AKT1,PRKDC,1


In [ ]:
"""
This protein is identified as Ribosomal protein S6 kinase alpha-1. The nucleotide sequence RPS6KA1 corresponds to the synthesis of this protein.
"""

In [4]:
import pandas as pd

slkb = pd.read_csv('../data/SLKB/SLKB_processed.csv')
genes = list(set(slkb['0']).union(set(slkb['1'])))
with open('../data/SLKB/gene.txt', 'w') as f:
    f.write(','.join([str(i) for i in genes]))

**Check data**

In [4]:
import os
import pandas as pd

has_func = [i.split('.')[0] for i in os.listdir('/data/qingyuyang/ESM4SL/data/SLKB/protrek_text_emb')]

cl_root = '/data/qingyuyang/ESM4SL/data/SLKB/cell_line/table'
for cl in os.listdir(cl_root):
    if cl.endswith('.csv') and not cl.startswith('overall'):
        print(f'Processing cell line {cl}') #.split('.')[0]
        df = pd.read_csv(os.path.join(cl_root, cl))
        genes = list(set(df['0']).union(set(df['1'])))
        for g in genes:
            if str(g) not in has_func:
                print(f'Missing gene {g}')

Processing cell line MELJUSO.csv
Missing gene 805
Missing gene 654364
Processing cell line HELA.csv
Processing cell line MEWO.csv
Missing gene 805
Missing gene 100526737
Missing gene 11037
Missing gene 56342
Missing gene 654364
Processing cell line JURKAT.csv
Processing cell line HSC5.csv
Missing gene 805
Missing gene 654364
Processing cell line A375.csv
Missing gene 100526737
Missing gene 11037
Missing gene 56342
Processing cell line PK1.csv
Missing gene 805
Missing gene 654364
Processing cell line 293T.csv
Processing cell line PC9.csv
Processing cell line A549.csv
Missing gene 805
Missing gene 654364
Processing cell line 786O.csv
Processing cell line MEL202.csv
Missing gene 805
Missing gene 654364
Processing cell line PATU8988S.csv
Missing gene 805
Missing gene 654364
Processing cell line 22RV1.csv
Processing cell line HS936T.csv
Missing gene 805
Missing gene 654364
Processing cell line HS944T.csv
Missing gene 805
Missing gene 654364
Processing cell line OVCAR8.csv
Processing cell li